In [1]:
import pandas as pd
from transformers import AutoTokenizer
import torch
import torch.nn.functional as F

# Load data
df = pd.read_csv('F:/workdir/Personal/dlgenaiproject_01/dlgenaiproject/dataset/train.csv')

f:\workdir\Personal\dlgenaiproject\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import AutoModelForSequenceClassification

# ── Label mapping ──────────────────────────────────────────────────────────────
ID2LABEL = {0: "A", 1: "B", 2: "C", 3: "D", 4: "E"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
NUM_LABELS = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ── DeBERTa-v3-small fine-tuned checkpoint ─────────────────────────────────────
DEBERTA_CKPT = "microsoft/deberta-v3-small"   # replace with fine-tuned path/hub-id

deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_CKPT)
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_CKPT,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
deberta_model.to(DEVICE)
deberta_model.eval()
print(f"DeBERTa loaded  : {DEBERTA_CKPT}")

# ── RoBERTa-base fine-tuned checkpoint ─────────────────────────────────────────
ROBERTA_CKPT = "roberta-base"                  # replace with fine-tuned path/hub-id

roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_CKPT)
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    ROBERTA_CKPT,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
roberta_model.to(DEVICE)
roberta_model.eval()
print(f"RoBERTa loaded  : {ROBERTA_CKPT}")


Using device: cuda


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 14570.00it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING  

DeBERTa loaded  : microsoft/deberta-v3-small


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4581.32it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RoBERTa loaded  : roberta-base


In [3]:
# ── Row 25 ─────────────────────────────────────────────────────────────────────
row = df.iloc[25]

# Build input: prompt + all five options
input_text = (
    f"{row['prompt']}\n"
    f"A: {row['A']}\n"
    f"B: {row['B']}\n"
    f"C: {row['C']}\n"
    f"D: {row['D']}\n"
    f"E: {row['E']}"
)

def run_inference(model, tokenizer, text, device):
    """Tokenize text, run model, return softmax probabilities."""
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits          # shape: (1, 5)
    probs = F.softmax(logits, dim=-1).squeeze()  # shape: (5,)
    return probs

# ── DeBERTa inference ──────────────────────────────────────────────────────────
deberta_probs = run_inference(deberta_model, deberta_tokenizer, input_text, DEVICE)
deberta_pred_id  = deberta_probs.argmax().item()
deberta_pred_lbl = ID2LABEL[deberta_pred_id]
deberta_pred_prob = deberta_probs[deberta_pred_id].item()

print("=== DeBERTa Probabilities ===")
for i, p in enumerate(deberta_probs):
    print(f"  {ID2LABEL[i]}: {p.item():.4f}")
print(f"\nQuestion 1 Answer → {deberta_pred_lbl}, {deberta_pred_prob:.4f}")

# ── RoBERTa inference ──────────────────────────────────────────────────────────
roberta_probs = run_inference(roberta_model, roberta_tokenizer, input_text, DEVICE)
roberta_pred_id  = roberta_probs.argmax().item()
roberta_pred_lbl = ID2LABEL[roberta_pred_id]
roberta_pred_prob = roberta_probs[roberta_pred_id].item()

print("\n=== RoBERTa Probabilities ===")
for i, p in enumerate(roberta_probs):
    print(f"  {ID2LABEL[i]}: {p.item():.4f}")
print(f"\nHighest probability option → {roberta_pred_lbl}, {roberta_pred_prob:.4f}")

# Question 1 Answer → {roberta_pred_lbl}, {roberta_pred_prob:.4f}
print(f"\nQuestion answer: {row['answer']}")


=== DeBERTa Probabilities ===
  A: 0.1940
  B: 0.1477
  C: 0.2307
  D: 0.2183
  E: 0.2094

Question 1 Answer → C, 0.2307

=== RoBERTa Probabilities ===
  A: 0.1869
  B: 0.2030
  C: 0.1975
  D: 0.2234
  E: 0.1891

Highest probability option → D, 0.2234

Question answer: E


In [4]:
# ── Simple probability ensemble (average) ─────────────────────────────────────
avg_probs = (deberta_probs + roberta_probs) / 2   # element-wise average

print("=== Ensemble (Averaged) Probabilities ===")
for i, p in enumerate(avg_probs):
    print(f"  {ID2LABEL[i]}: {p.item():.4f}")

ensemble_pred_id   = avg_probs.argmax().item()
ensemble_pred_lbl  = ID2LABEL[ensemble_pred_id]
ensemble_pred_prob = avg_probs[ensemble_pred_id].item()

print(f"\nQuestion 2 Answer → {ensemble_pred_lbl}, {ensemble_pred_prob:.4f}")
print(f"Ground-truth answer: {row['answer']}")


=== Ensemble (Averaged) Probabilities ===
  A: 0.1905
  B: 0.1753
  C: 0.2141
  D: 0.2208
  E: 0.1992

Question 2 Answer → D, 0.2208
Ground-truth answer: E


In [5]:
# ── Q3 + Q4 : Weighted Ensemble on row 25 ─────────────────────────────────────
# P(final) = 0.7 × P(DeBERTa) + 0.3 × P(RoBERTa)
weighted_probs_r25 = 0.7 * deberta_probs + 0.3 * roberta_probs

print("=== Weighted Ensemble Probabilities (0.7×DeBERTa + 0.3×RoBERTa) ===")
for i, p in enumerate(weighted_probs_r25):
    print(f"  {ID2LABEL[i]}: {p.item():.4f}")

ranked_ids_r25  = weighted_probs_r25.argsort(descending=True).tolist()
ranked_lbls_r25 = [ID2LABEL[idx] for idx in ranked_ids_r25]

print(f"\nQuestion 3 → Top-1: {ranked_lbls_r25[0]}, {weighted_probs_r25[ranked_ids_r25[0]].item():.4f}")

# Q4 : Top-3 Kaggle format
top3_r25 = " ".join(ranked_lbls_r25[:3])
print(f"Question 4 → Top-3 string: {top3_r25}")
print(f"Ground-truth answer: {row['answer']}")


=== Weighted Ensemble Probabilities (0.7×DeBERTa + 0.3×RoBERTa) ===
  A: 0.1918
  B: 0.1643
  C: 0.2208
  D: 0.2199
  E: 0.2032

Question 3 → Top-1: C, 0.2208
Question 4 → Top-3 string: C D E
Ground-truth answer: E


In [6]:
# ── Q5 : Full test.csv → weighted ensemble → submission.csv ───────────────────

test_df = pd.read_csv('F:/workdir/Personal/dlgenaiproject_01/dlgenaiproject/dataset/test.csv')

def build_input(r):
    return (
        f"{r['prompt']}\n"
        f"A: {r['A']}\nB: {r['B']}\nC: {r['C']}\nD: {r['D']}\nE: {r['E']}"
    )

# Store per-row results for reuse in Q7-Q9
test_records = []   # list of dicts: id, deberta_probs, weighted_probs

print(f"Running inference on {len(test_df)} test rows …")
for idx, r in test_df.iterrows():
    text  = build_input(r)
    d_p   = run_inference(deberta_model,  deberta_tokenizer,  text, DEVICE)
    rob_p = run_inference(roberta_model,  roberta_tokenizer,  text, DEVICE)
    w_p   = 0.7 * d_p + 0.3 * rob_p
    test_records.append({
        "id":             r["id"],
        "deberta_probs":  d_p,
        "weighted_probs": w_p,
    })
    if (idx + 1) % 50 == 0:
        print(f"  … {idx + 1}/{len(test_df)} done")

# Build submission DataFrame (Top-3 from weighted ensemble)
submission_rows = []
for rec in test_records:
    ranked = rec["weighted_probs"].argsort(descending=True).tolist()
    top3   = " ".join([ID2LABEL[i] for i in ranked[:3]])
    submission_rows.append({"id": rec["id"], "prediction": top3})

sub_df = pd.DataFrame(submission_rows)
sub_df.to_csv(
    'F:/workdir/Personal/dlgenaiproject_01/dlgenaiproject/milestone/submission.csv',
    index=False
)

print(f"\nSubmission saved.")
print(f"Question 5 → Prediction rows (excluding header): {len(sub_df)}")


Running inference on 500 test rows …
  … 50/500 done
  … 100/500 done
  … 150/500 done
  … 200/500 done
  … 250/500 done
  … 300/500 done
  … 350/500 done
  … 400/500 done
  … 450/500 done
  … 500/500 done

Submission saved.
Question 5 → Prediction rows (excluding header): 500


In [7]:
# ── Q6 : Test-Time Augmentation on first 50 test rows (DeBERTa only) ──────────
TTA_PREFIX = "Answer the following multiple-choice question carefully: "

tta_diff_count = 0

for rec_idx in range(50):
    r = test_df.iloc[rec_idx]

    # Original prompt
    orig_text = build_input(r)
    p_orig = run_inference(deberta_model, deberta_tokenizer, orig_text, DEVICE)

    # Instruction-augmented prompt
    aug_row  = r.copy()
    aug_row["prompt"] = TTA_PREFIX + r["prompt"]
    aug_text = build_input(aug_row)
    p_aug = run_inference(deberta_model, deberta_tokenizer, aug_text, DEVICE)

    # Average the two passes
    p_tta = (p_orig + p_aug) / 2

    top1_orig = ID2LABEL[p_orig.argmax().item()]
    top1_tta  = ID2LABEL[p_tta.argmax().item()]

    if top1_orig != top1_tta:
        tta_diff_count += 1

print(f"Question 6 → Rows with different Top-1 after TTA (first 50): {tta_diff_count}")


Question 6 → Rows with different Top-1 after TTA (first 50): 0


In [ ]:
# ── Q7 + Q8 : Analysis on first 100 test rows ────────────────────────────────
# Reuses test_records populated in cell 6 (Q5)

q7_diff     = 0   # Q7 : different Top-1 between DeBERTa and Weighted Ensemble
q8_pos_gain = 0   # Q8 : rows where ensemble max-prob > deberta max-prob

for rec in test_records[:100]:
    d_p = rec["deberta_probs"]
    w_p = rec["weighted_probs"]

    # ── Q7 : Top-1 comparison ─────────────────────────────────────────────────
    if ID2LABEL[d_p.argmax().item()] != ID2LABEL[w_p.argmax().item()]:
        q7_diff += 1

    # ── Q8 : Confidence gain ──────────────────────────────────────────────────
    if w_p.max().item() - d_p.max().item() > 0:
        q8_pos_gain += 1

print(f"Question 7 → Different Top-1 (DeBERTa vs Weighted Ensemble, first 100): {q7_diff}")
print(f"Question 8 → Rows with positive confidence gain (first 100)           : {q8_pos_gain}")


Question 7 → Different Top-1 (DeBERTa vs Weighted Ensemble, first 100): 18
Question 8 → Rows with positive confidence gain (first 100)           : 0
Question 9 → Rows with at least one Top-3 change (first 100)          : 18


In [10]:
# ── Q9 : Top-3 ordered ranking comparison — DeBERTa vs Weighted Ensemble ──────
# First 100 rows of test.csv
# Reuses test_records populated in cell 6 (Q5)
#
# A change is counted when the ordered Top-3 string differs at all,
# e.g.  "A C D"  vs  "A D C"  → counted as different

q9_top3_diff  = 0
q9_detail     = []   # store (row_id, d_top3, w_top3) only for changed rows

for rec in test_records[:100]:
    d_p = rec["deberta_probs"]
    w_p = rec["weighted_probs"]

    d_top3 = " ".join([ID2LABEL[i] for i in d_p.argsort(descending=True)[:3].tolist()])
    w_top3 = " ".join([ID2LABEL[i] for i in w_p.argsort(descending=True)[:3].tolist()])

    if d_top3 != w_top3:
        q9_top3_diff += 1
        q9_detail.append((rec["id"], d_top3, w_top3))

# Show first 10 changed rows as examples
print(f"{'ID':<6}  {'DeBERTa Top-3':<15}  {'Ensemble Top-3':<15}")
print("-" * 42)
for row_id, d, w in q9_detail[:10]:
    print(f"{row_id:<6}  {d:<15}  {w:<15}")

print(f"\nQuestion 9 → Rows with at least one Top-3 change (first 100): {q9_top3_diff}")


ID      DeBERTa Top-3    Ensemble Top-3 
------------------------------------------
4       C D E            D C E          
13      C D E            D C E          
31      C D E            D C E          
34      C D E            D C E          
38      C D E            D C E          
43      C D E            D C E          
46      C D E            D C E          
49      C D E            D C E          
50      C D E            D C E          
59      C D E            D C E          

Question 9 → Rows with at least one Top-3 change (first 100): 18


In [9]:
# ── Q10 : MAP@3 on first 100 validation rows (train.csv has ground truth) ─────
# AP@3 for a single correct answer = 1/rank  if correct answer is in Top-3
#                                  = 0       otherwise
# MAP@3 = mean of AP@3 across all rows

ap_scores = []

for _, r in df.head(100).iterrows():
    text  = build_input(r)
    d_p   = run_inference(deberta_model,  deberta_tokenizer,  text, DEVICE)
    rob_p = run_inference(roberta_model,  roberta_tokenizer,  text, DEVICE)
    w_p   = 0.7 * d_p + 0.3 * rob_p

    top3  = [ID2LABEL[i] for i in w_p.argsort(descending=True)[:3].tolist()]
    truth = r["answer"]

    # Find rank of the correct answer in Top-3 (1-indexed)
    ap = 0.0
    for k, pred in enumerate(top3, start=1):
        if pred == truth:
            ap = 1.0 / k
            break
    ap_scores.append(ap)

map3 = sum(ap_scores) / len(ap_scores)
print(f"Question 10 → MAP@3 (first 100 validation rows, weighted ensemble): {map3:.4f}")


Question 10 → MAP@3 (first 100 validation rows, weighted ensemble): 0.3533
